In [ ]:
# Multi-Commodity Flow — source/sink (fractional) formulation — gurobipy
import gurobipy as gp
from gurobipy import GRB
V=[1,2,3,4]; arcs=[(1,2),(1,3),(2,3),(2,4),(3,4)]
cap={(1,2):10,(1,3):8,(2,3):4,(2,4):7,(3,4):9}
cost={(1,2):1,(1,3):2,(2,3):1,(2,4):3,(3,4):1}
K={1:(1,4,10),2:(1,4,5)}
m=gp.Model("MCF_SourceSink")
f=m.addVars([(i,u,v) for i in K for (u,v) in arcs], lb=0, ub=1, name="f")
m.setObjective(gp.quicksum(cost[u,v]*K[i][2]*f[i,u,v] for i in K for (u,v) in arcs), GRB.MINIMIZE)
for (u,v) in arcs:
    m.addConstr(gp.quicksum(K[i][2]*f[i,u,v] for i in K) <= cap[u,v], f"Cap_{u}_{v}")
for i,(s,t,d) in K.items():
    for u in V:
        out_=gp.quicksum(f[i,u,w] for (x,w) in arcs if x==u)
        in_ =gp.quicksum(f[i,w,u] for (w,x) in arcs if x==u)
        m.addConstr(out_-in_ == (1 if u==s else -1 if u==t else 0), f"Bal_{i}_{u}")
m.optimize()
if m.status==GRB.OPTIMAL:
    for i in K:
        print(f"commodity {i}:", {f"{u}->{v}":round(f[i,u,v].X,3) for (u,v) in arcs if f[i,u,v].X>1e-6})
    print("Total cost =", m.ObjVal)
